# Bayesian SSE Association Models

In [1]:
from __future__ import annotations

from collections.abc import Iterable, Sequence
from pathlib import Path
import re
import sys

import arviz as az
import bambi as bmb
from IPython.display import display
import numpy as np
import pandas as pd
from scipy.special import logit as logit_func

%load_ext autoreload
%autoreload 2

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "utils").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "utils").exists():
    raise RuntimeError("Run this notebook from inside the scotland repository.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from sse_detection.lib import (  # noqa: E402
    DEFAULT_MIXING_FEATURES,
    add_observed_mixing_entropy_scales,
    HIGH_PRIORITY_CANDIDATE_TIERS,
    load_sse_outputs,
    load_sequence_data,
)

SSE_OUTPUT_DIR = PROJECT_ROOT / "sse_detection" / "results" / "sse_outputs"

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

In [ ]:
STANDARDISE_SPECS = {
    "z_wn_prop_sequenced": "wn_prop_sequenced",
    "z_log1p_wn_positive_tests": "log1p_wn_positive_tests",
    "z_dz_cum_prop_sequenced": "dz_cum_prop_sequenced",
    "z_dz_cum_incidence_per_capita": "dz_cum_incidence_per_capita",
    "z_dz_7d_test_positivity": "dz_7d_test_positivity",
    "z_log1p_dz_cum_positive_tests": "log1p_dz_cum_positive_tests",
    "z_dz_cum_prop_vaccinated": "dz_cum_prop_vaccinated",
}

COMPOSITION_SPECS = [
    {
        "name": "sex",
        "column": "sex",
        "reference": "Male",
        "label": "Sex",
    },
    {
        "name": "age_band",
        "column": "age_band",
        "reference": "20-24",
        "label": "Age band",
    },
    {
        "name": "simd_quintile",
        "column": "dz_simd_quintile",
        "reference": "1",
        "label": "SIMD quintile",
    },
    {
        "name": "urban_rural_class",
        "column": "dz_urban_rural_class",
        "reference": "Large Urban Areas",
        "label": "Urban/rural class",
    },
    {
        "name": "health_board",
        "column": "dz_health_board",
        "reference": "Greater Glasgow and Clyde",
        "label": "Health board",
    },
]

EXPANDED_CONTEXT_ADJUSTERS = [
    "z_dz_cum_prop_sequenced",
    "z_dz_cum_incidence_per_capita",
    "z_dz_7d_test_positivity",
    "z_log1p_dz_cum_positive_tests",
]


def add_standardised_adjusters(data: pd.DataFrame) -> pd.DataFrame:
    """Add standardised surveillance and context adjusters used by notebooks."""
    out = add_observed_mixing_entropy_scales(data)
    if "wn_positive_tests" in out.columns:
        out["log1p_wn_positive_tests"] = np.log1p(out["wn_positive_tests"])
    if "dz_cum_positive_tests" in out.columns:
        out["log1p_dz_cum_positive_tests"] = np.log1p(out["dz_cum_positive_tests"])
    for target, source in STANDARDISE_SPECS.items():
        if source not in out.columns:
            continue
        values = out[source].astype(float)
        sd = values.std(skipna=True)
        if pd.isna(sd) or sd == 0:
            out[target] = np.nan
        else:
            out[target] = (values - values.mean(skipna=True)) / sd
    return out

In [ ]:
RANDOM_SEED = 123
CLUSTER_ID_COL = "cluster_id"
MIXING_GROUP_VARS = ("window_idx", "clade")
COMP_GROUP_VARS = MIXING_GROUP_VARS + (CLUSTER_ID_COL,)

COMPOSITION_PREDICTORS = {
    spec["column"]: spec["reference"] for spec in COMPOSITION_SPECS
}
MIXING_PREDICTORS = list(DEFAULT_MIXING_FEATURES)
EXPANDED_ADJUSTERS = list(EXPANDED_CONTEXT_ADJUSTERS)

DEV_SAMPLE_ROWS = 1_000

# The full Bayesian grid is computationally expensive. Keep the example
# model on by default, then opt into the full grid when needed.
RUN_HEALTH_BOARD_EXAMPLE = True
RUN_MIXING_GRID = False

SAMPLING_KWARGS = {
    "draws": 2_000,
    "tune": 2_000,
    "chains": 4,
    "cores": 4,
    "target_accept": 0.99,
    "random_seed": RANDOM_SEED,
}

## Load and Align Data

The node outcome follows the main pipeline: high-priority burst or burden candidates among nodes at least as large as the smallest high-priority candidate. Sequence-level models inherit the candidate label from their cluster.


In [ ]:
sse_outputs = load_sse_outputs(SSE_OUTPUT_DIR)
cluster_data = add_standardised_adjusters(sse_outputs.cluster_table.copy())
sequence_data = add_standardised_adjusters(load_sequence_data())

cluster_data["candidate"] = cluster_data["candidate_tier"].isin(
    HIGH_PRIORITY_CANDIDATE_TIERS
)

candidate_sizes = cluster_data.loc[cluster_data["candidate"], "cluster_size"].dropna()
if candidate_sizes.empty:
    raise ValueError("No high-priority candidate nodes were found.")

min_candidate_size = int(candidate_sizes.min())
eligible_nodes = cluster_data.loc[
    cluster_data["cluster_size"].ge(min_candidate_size)
].copy()

eligible_sequence_data = sequence_data.merge(
    eligible_nodes[[CLUSTER_ID_COL, "candidate"]],
    on=CLUSTER_ID_COL,
    how="inner",
)

candidate_node_rate = float(eligible_nodes["candidate"].mean())
candidate_sequence_rate = float(eligible_sequence_data["candidate"].mean())

candidate_summary = pd.DataFrame(
    [
        {
            "dataset": "eligible_nodes",
            "rows": len(eligible_nodes),
            "candidate_rate": candidate_node_rate,
            "candidates": int(eligible_nodes["candidate"].sum()),
        },
        {
            "dataset": "eligible_sequence_data",
            "rows": len(eligible_sequence_data),
            "candidate_rate": candidate_sequence_rate,
            "candidates": int(eligible_sequence_data["candidate"].sum()),
        },
    ]
)
display(candidate_summary)

## Data and Formula Helpers


In [ ]:
def _unique_preserve_order(items: Iterable[str]) -> list[str]:
    """Return unique strings in first-seen order."""
    out: list[str] = []
    seen: set[str] = set()
    for item in items:
        if item not in seen:
            out.append(item)
            seen.add(item)
    return out


def _categories_from(series: pd.Series) -> list:
    """Use declared categories when present; otherwise preserve observed order."""
    if isinstance(series.dtype, pd.CategoricalDtype):
        return list(series.cat.categories)
    return series.dropna().drop_duplicates().tolist()


def sample_model_test_data(
    data: pd.DataFrame,
    *,
    outcome: str = "candidate",
    max_rows: int = DEV_SAMPLE_ROWS,
    positive_fraction: float = 0.35,
    random_state: int = RANDOM_SEED,
    categorical_vars: Sequence[str] | None = None,
) -> pd.DataFrame:
    """Return a balanced-ish development sample while preserving categories."""
    positives = data.loc[data[outcome] == 1]
    negatives = data.loc[data[outcome] == 0]

    if positives.empty or negatives.empty:
        raise ValueError(f"'{outcome}' must contain at least one 0 and one 1.")

    positive_fraction = float(np.clip(positive_fraction, 0.01, 0.99))
    n_pos = min(len(positives), max(1, int(round(max_rows * positive_fraction))))
    n_neg = min(len(negatives), max_rows - n_pos)
    if n_neg <= 0:
        raise ValueError("Sampling settings left no room for negative controls.")

    pos_sample = positives.sample(
        n=n_pos,
        random_state=random_state,
        replace=False,
    )
    neg_sample = negatives.sample(
        n=n_neg,
        random_state=random_state,
        replace=False,
    )

    sampled = pd.concat([pos_sample, neg_sample], axis=0)
    sampled = sampled.sample(frac=1, random_state=random_state).reset_index(drop=True)

    if categorical_vars is not None:
        for col in categorical_vars:
            if col in sampled.columns:
                sampled[col] = pd.Categorical(
                    sampled[col],
                    categories=_categories_from(data[col]),
                )

    return sampled


def get_complete_case_data(
    df: pd.DataFrame,
    *,
    outcome: str = "candidate",
    predictors: Iterable[str] | None = None,
    group_vars: Sequence[str] = GROUP_VARS,
    categorical_vars: Sequence[str] = (),
    id_cols: Sequence[str] = (CLUSTER_ID_COL,),
    verbose: bool = True,
) -> pd.DataFrame:
    """Create a complete-case model frame with stable categorical dtypes."""
    if predictors is None:
        raise ValueError("Please provide predictor columns.")

    required_cols = _unique_preserve_order(
        [outcome, *id_cols, *list(predictors), *group_vars]
    )
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing columns in dataframe: {missing_cols}")

    before_n = len(df)
    out = df.loc[:, required_cols].dropna().copy()
    after_n = len(out)

    out[outcome] = out[outcome].astype(int)

    categorical_cols = _unique_preserve_order(
        [*categorical_vars, *group_vars, *id_cols]
    )
    for col in categorical_cols:
        if col in out.columns:
            out[col] = pd.Categorical(out[col], categories=_categories_from(df[col]))

    if verbose:
        print("Complete-case summary")
        print("---------------------")
        print(f"Rows before dropna: {before_n:,}")
        print(f"Rows after dropna:  {after_n:,}")
        print(f"Rows dropped:       {before_n - after_n:,}")
        print(f"Percent retained:   {after_n / before_n:.1%}")
        print(f"Candidate rate:     {out[outcome].mean():.2%}")
        print("\nGrouping levels:")
        for col in group_vars:
            print(f"{col}: {out[col].nunique()} levels")

    return out


def treatment_term(variable: str, reference: str | int | float | None) -> str:
    """Return a Bambi/formulae categorical term with an optional reference level."""
    if reference is None:
        return f"C({variable})"
    return f"C({variable}, Treatment(reference={reference!r}))"


def formula_with_varying_intercepts(
    outcome: str,
    terms: str | Sequence[str],
    *,
    group_vars: Sequence[str] = GROUP_VARS,
) -> str:
    """Build a logistic mixed-model formula with window/clade intercepts."""
    if isinstance(terms, str):
        terms = [terms]
    rhs_terms = [*terms, *(f"(1|{group})" for group in group_vars)]
    return f"{outcome} ~ " + " + ".join(rhs_terms)

## Bambi Fitting and Summaries


In [ ]:
_RANDOM_EFFECT_RE = re.compile(r"\([^|()]+\|([^()]+)\)")


def random_effect_groups(formula: str) -> list[str]:
    """Extract grouping variables from terms such as ``(1|window_idx)``."""
    return [match.strip() for match in _RANDOM_EFFECT_RE.findall(formula)]


def fit_bayesian_logistic_model(
    data: pd.DataFrame,
    formula: str,
    *,
    family: str = "bernoulli",
    categorical: Sequence[str] | None = None,
    fixed_prior_sigma: float = 1.0,
    intercept_prior_sigma: float = 1.5,
    random_effect_sigma: float = 1.0,
    draws: int = 2_000,
    tune: int = 2_000,
    chains: int = 4,
    cores: int = 4,
    target_accept: float = 0.99,
    random_seed: int = RANDOM_SEED,
    log_likelihood: bool = True,
    noncentered: bool = True,
) -> tuple[bmb.Model, az.InferenceData]:
    """Fit a Bambi hierarchical logistic regression with weakly informative priors."""
    if "~" not in formula:
        raise ValueError("Formula must contain '~', e.g. 'candidate ~ x + (1|group)'.")

    response = formula.split("~", 1)[0].strip()
    if response not in data.columns:
        raise ValueError(f"Response variable '{response}' not found in data.")

    outcome_mean = float(np.clip(data[response].mean(), 1e-6, 1 - 1e-6))
    priors = {
        "Intercept": bmb.Prior(
            "Normal",
            mu=logit_func(outcome_mean),
            sigma=intercept_prior_sigma,
        ),
        "common": bmb.Prior("Normal", mu=0, sigma=fixed_prior_sigma),
        "group_specific": bmb.Prior(
            "Normal",
            mu=0,
            sigma=bmb.Prior("HalfNormal", sigma=random_effect_sigma),
        ),
    }

    group_cols = random_effect_groups(formula)
    categorical_cols = _unique_preserve_order([*(categorical or ()), *group_cols])

    model = bmb.Model(
        formula=formula,
        data=data,
        family=family,
        priors=priors,
        categorical=categorical_cols or None,
        noncentered=noncentered,
    )
    idata = model.fit(
        draws=draws,
        tune=tune,
        chains=chains,
        cores=cores,
        target_accept=target_accept,
        random_seed=random_seed,
        idata_kwargs={"log_likelihood": log_likelihood},
    )
    return model, idata


def _print_section(title: str, char: str = "=") -> None:
    print(f"\n{title}")
    print(char * len(title))


def _format_df_for_print(
    df: pd.DataFrame,
    *,
    float_digits: int = 4,
    width: int = 160,
    max_colwidth: int = 80,
) -> str:
    with pd.option_context(
        "display.max_rows",
        None,
        "display.max_columns",
        None,
        "display.width",
        width,
        "display.max_colwidth",
        max_colwidth,
        "display.float_format",
        lambda x: f"{x:,.{float_digits}f}",
    ):
        return df.to_string()


def _show_table(
    df: pd.DataFrame,
    *,
    display_tables: bool = False,
    float_digits: int = 4,
) -> None:
    if display_tables:
        display(df.style.format(precision=float_digits))
    else:
        print(_format_df_for_print(df, float_digits=float_digits))


def _available_posterior_vars(
    idata: az.InferenceData,
    var_names: Sequence[str] | None,
) -> list[str] | None:
    if var_names is None:
        return None
    available = set(idata.posterior.data_vars)
    selected = [var for var in var_names if var in available]
    missing = [var for var in var_names if var not in available]
    if missing:
        print("Skipping unavailable posterior variables:")
        for var in missing:
            print(f"  - {var}")
    if not selected:
        raise KeyError("None of the requested posterior variables were found.")
    return selected


def summarise_bambi_idata(
    idata: az.InferenceData,
    *,
    var_names: Sequence[str] | None = None,
    hdi_prob: float = 0.95,
    odds_ratio_vars: Sequence[str] | None = None,
    print_diagnostics: bool = True,
    rhat_threshold: float = 1.01,
    ess_threshold: int = 400,
    display_tables: bool = False,
    float_digits: int = 4,
) -> pd.DataFrame:
    """Print diagnostics and return a focused ArviZ posterior summary."""
    selected_vars = _available_posterior_vars(idata, var_names)
    summary_df = az.summary(
        idata,
        var_names=selected_vars,
        hdi_prob=hdi_prob,
        round_to=float_digits,
    )
    diagnostic_summary = az.summary(
        idata,
        hdi_prob=hdi_prob,
        round_to=float_digits,
    )

    if print_diagnostics:
        _print_section("Bayesian model diagnostics")
        diagnostic_rows = []

        if hasattr(idata, "sample_stats") and "diverging" in idata.sample_stats:
            n_div = int(idata.sample_stats["diverging"].sum().item())
            n_total = int(idata.sample_stats["diverging"].size)
            div_rate = n_div / n_total
            diagnostic_rows.append(
                {
                    "Diagnostic": "Divergences",
                    "Value": f"{n_div} / {n_total} ({div_rate:.2%})",
                    "Status": "OK" if n_div == 0 else "WARNING",
                    "Interpretation": (
                        "No divergent transitions."
                        if n_div == 0
                        else "Investigate divergent transitions."
                    ),
                }
            )

        try:
            bfmi = np.asarray(az.bfmi(idata))
            min_bfmi = float(np.nanmin(bfmi))
            bfmi_by_chain = ", ".join(f"{x:.3f}" for x in bfmi)
            diagnostic_rows.append(
                {
                    "Diagnostic": "BFMI",
                    "Value": f"min={min_bfmi:.3f}; chains=[{bfmi_by_chain}]",
                    "Status": "OK" if min_bfmi >= 0.3 else "WARNING",
                    "Interpretation": (
                        "Energy exploration looks acceptable."
                        if min_bfmi >= 0.3
                        else "One or more chains have BFMI < 0.3."
                    ),
                }
            )
        except Exception as exc:
            diagnostic_rows.append(
                {
                    "Diagnostic": "BFMI",
                    "Value": "Could not compute",
                    "Status": "NA",
                    "Interpretation": str(exc),
                }
            )

        if "r_hat" in diagnostic_summary.columns:
            max_rhat = float(diagnostic_summary["r_hat"].max(skipna=True))
            diagnostic_rows.append(
                {
                    "Diagnostic": "Max R-hat",
                    "Value": f"{max_rhat:.4f}",
                    "Status": "OK" if max_rhat <= rhat_threshold else "WARNING",
                    "Interpretation": (
                        f"All posterior variables are at or below {rhat_threshold}."
                        if max_rhat <= rhat_threshold
                        else f"Some posterior variables exceed {rhat_threshold}."
                    ),
                }
            )

        if "ess_bulk" in diagnostic_summary.columns:
            min_bulk_ess = float(diagnostic_summary["ess_bulk"].min(skipna=True))
            diagnostic_rows.append(
                {
                    "Diagnostic": "Min bulk ESS",
                    "Value": f"{min_bulk_ess:.1f}",
                    "Status": "OK" if min_bulk_ess >= ess_threshold else "WARNING",
                    "Interpretation": (
                        f"All bulk ESS values are at least {ess_threshold}."
                        if min_bulk_ess >= ess_threshold
                        else f"Some bulk ESS values are below {ess_threshold}."
                    ),
                }
            )

        if "ess_tail" in diagnostic_summary.columns:
            min_tail_ess = float(diagnostic_summary["ess_tail"].min(skipna=True))
            diagnostic_rows.append(
                {
                    "Diagnostic": "Min tail ESS",
                    "Value": f"{min_tail_ess:.1f}",
                    "Status": "OK" if min_tail_ess >= ess_threshold else "WARNING",
                    "Interpretation": (
                        f"All tail ESS values are at least {ess_threshold}."
                        if min_tail_ess >= ess_threshold
                        else f"Some tail ESS values are below {ess_threshold}."
                    ),
                }
            )

        if hasattr(idata, "sample_stats") and "tree_depth" in idata.sample_stats:
            max_tree_depth = int(idata.sample_stats["tree_depth"].max().item())
            diagnostic_rows.append(
                {
                    "Diagnostic": "Max tree depth",
                    "Value": str(max_tree_depth),
                    "Status": "INFO",
                    "Interpretation": "Maximum observed tree depth.",
                }
            )

        _show_table(
            pd.DataFrame(diagnostic_rows),
            display_tables=display_tables,
            float_digits=float_digits,
        )

        _print_section("Posterior summary")
        _show_table(
            summary_df,
            display_tables=display_tables,
            float_digits=float_digits,
        )

    if odds_ratio_vars is not None:
        _print_section("Odds-ratio summaries")
        available = set(idata.posterior.data_vars)
        for var in odds_ratio_vars:
            if var not in available:
                print(f"\n{var}: not found in idata.posterior")
                continue

            beta = idata.posterior[var]
            odds_ratio = np.exp(beta)
            or_idata = odds_ratio.to_dataset(name=f"OR_{var}")
            or_summary = az.summary(
                or_idata,
                hdi_prob=hdi_prob,
                round_to=float_digits,
            )

            prob_df = pd.DataFrame(
                {
                    "Quantity": [
                        "P(beta > 0 | data)",
                        "P(beta < 0 | data)",
                        "P(OR > 1 | data)",
                        "P(OR < 1 | data)",
                    ],
                    "Probability": [
                        float((beta > 0).mean().item()),
                        float((beta < 0).mean().item()),
                        float((odds_ratio > 1).mean().item()),
                        float((odds_ratio < 1).mean().item()),
                    ],
                }
            )

            _print_section(var, char="-")
            _show_table(
                or_summary,
                display_tables=display_tables,
                float_digits=float_digits,
            )
            _show_table(
                prob_df,
                display_tables=display_tables,
                float_digits=float_digits,
            )

    return summary_df


def fit_and_summarise_model(
    data: pd.DataFrame,
    formula: str,
    *,
    var_names: Sequence[str],
    odds_ratio_vars: Sequence[str],
    categorical: Sequence[str] = GROUP_VARS,
    display_tables: bool = True,
    **fit_kwargs,
) -> dict[str, object]:
    """Fit a model and keep the model, posterior, formula, and focused summary."""
    model, idata = fit_bayesian_logistic_model(
        data=data,
        formula=formula,
        categorical=categorical,
        **fit_kwargs,
    )
    summary = summarise_bambi_idata(
        idata,
        var_names=var_names,
        hdi_prob=0.95,
        odds_ratio_vars=odds_ratio_vars,
        display_tables=display_tables,
    )
    return {
        "formula": formula,
        "model": model,
        "idata": idata,
        "summary": summary,
    }

## Composition Models: Sequence-Level Association

Composition models ask whether individual sequence attributes are associated with membership in a candidate cluster, with varying intercepts for window and clade.


In [ ]:
composition_terms = {
    column: treatment_term(column, reference)
    for column, reference in COMPOSITION_PREDICTORS.items()
}

composition_model_df = get_complete_case_data(
    df=eligible_sequence_data,
    outcome="candidate",
    predictors=COMPOSITION_PREDICTORS.keys(),
    group_vars=COMP_GROUP_VARS,
    categorical_vars=(*COMP_GROUP_VARS, *COMPOSITION_PREDICTORS.keys()),
)

composition_formulas = {
    column: formula_with_varying_intercepts("candidate", term, group_vars=COMP_GROUP_VARS)
    for column, term in composition_terms.items()
}
joint_composition_formula = formula_with_varying_intercepts(
    "candidate",
    list(composition_terms.values()),
    group_vars=COMP_GROUP_VARS
)

display(
    pd.DataFrame(
        [
            {"model": column, "formula": formula}
            for column, formula in composition_formulas.items()
        ]
        + [{"model": "joint_composition", "formula": joint_composition_formula}]
    )
)

In [ ]:
health_board_col = "dz_health_board"
health_board_term = composition_terms[health_board_col]
health_board_formula = composition_formulas[health_board_col]

seq_subset = sample_model_test_data(
    composition_model_df,
    max_rows=DEV_SAMPLE_ROWS,
    positive_fraction=candidate_sequence_rate,
    categorical_vars=[
        *COMPOSITION_PREDICTORS.keys(),
        *COMP_GROUP_VARS,
        CLUSTER_ID_COL,
    ],
)

print(health_board_formula)
print(f"Rows in sampled model frame: {len(seq_subset):,}")
print(f"Sample candidate rate: {seq_subset['candidate'].mean():.2%}")

if RUN_HEALTH_BOARD_EXAMPLE:
    health_board_results = fit_and_summarise_model(
        data=seq_subset,
        formula=health_board_formula,
        var_names=[
            "Intercept",
            health_board_term,
            "1|window_idx_sigma",
            "1|clade_sigma",
            "1|cluster_id_sigma",
        ],
        odds_ratio_vars=[health_board_term],
        categorical=COMP_GROUP_VARS,
        **SAMPLING_KWARGS,
    )
else:
    health_board_results = None
    print("RUN_HEALTH_BOARD_EXAMPLE=False; model was prepared but not sampled.")


## Mixing Models: Node-Level Association

Mixing models ask whether candidate nodes have unusual cluster-level entropy or context profiles, again with varying intercepts for window and clade.


In [ ]:
mixing_primary_df = get_complete_case_data(
    df=eligible_nodes,
    outcome="candidate",
    predictors=MIXING_PREDICTORS,
    group_vars=MIXING_GROUP_VARS,
    categorical_vars=MIXING_GROUP_VARS,
)

mixing_expanded_df = get_complete_case_data(
    df=eligible_nodes,
    outcome="candidate",
    predictors=[*MIXING_PREDICTORS, *EXPANDED_ADJUSTERS],
    group_vars=MIXING_GROUP_VARS,
    categorical_vars=MIXING_GROUP_VARS,
)

mixing_single_primary_formulas = {
    col: formula_with_varying_intercepts("candidate", col) for col in MIXING_PREDICTORS
}
mixing_single_expanded_formulas = {
    col: formula_with_varying_intercepts(
        "candidate",
        [col, *EXPANDED_ADJUSTERS],
    )
    for col in MIXING_PREDICTORS
}
joint_mixing_primary_formula = formula_with_varying_intercepts(
    "candidate",
    MIXING_PREDICTORS,
)
joint_mixing_expanded_formula = formula_with_varying_intercepts(
    "candidate",
    [*MIXING_PREDICTORS, *EXPANDED_ADJUSTERS],
)

display(
    pd.DataFrame(
        [
            {"model": "single_primary", "predictor": col, "formula": formula}
            for col, formula in mixing_single_primary_formulas.items()
        ]
        + [
            {"model": "single_expanded", "predictor": col, "formula": formula}
            for col, formula in mixing_single_expanded_formulas.items()
        ]
        + [
            {
                "model": "joint_primary",
                "predictor": "all_mixing_predictors",
                "formula": joint_mixing_primary_formula,
            },
            {
                "model": "joint_expanded",
                "predictor": "all_mixing_predictors",
                "formula": joint_mixing_expanded_formula,
            },
        ]
    )
)


In [ ]:
single_primary_results: dict[str, dict[str, object]] = {}

if RUN_MIXING_GRID:
    for col, formula in mixing_single_primary_formulas.items():
        print(f"\nFitting primary mixing model: {col}")
        single_primary_results[col] = fit_and_summarise_model(
            data=mixing_primary_df,
            formula=formula,
            var_names=[
                "Intercept",
                col,
                "1|window_idx_sigma",
                "1|clade_sigma",
            ],
            odds_ratio_vars=[col],
            categorical=MIXING_GROUP_VARS,
            **SAMPLING_KWARGS,
        )
else:
    print("RUN_MIXING_GRID=False; skipping primary single-predictor sampling.")


In [ ]:
single_expanded_results: dict[str, dict[str, object]] = {}

if RUN_MIXING_GRID:
    for col, formula in mixing_single_expanded_formulas.items():
        print(f"\nFitting expanded mixing model: {col}")
        single_expanded_results[col] = fit_and_summarise_model(
            data=mixing_expanded_df,
            formula=formula,
            var_names=[
                "Intercept",
                col,
                *EXPANDED_ADJUSTERS,
                "1|window_idx_sigma",
                "1|clade_sigma",
            ],
            odds_ratio_vars=[col],
            categorical=MIXING_GROUP_VARS,
            **SAMPLING_KWARGS,
        )
else:
    print("RUN_MIXING_GRID=False; skipping expanded single-predictor sampling.")


In [ ]:
joint_primary_results: dict[str, dict[str, object]] = {}
joint_expanded_results: dict[str, dict[str, object]] = {}

if RUN_MIXING_GRID:
    print("\nFitting joint primary mixing model")
    joint_primary_results["joint_primary"] = fit_and_summarise_model(
        data=mixing_primary_df,
        formula=joint_mixing_primary_formula,
        var_names=[
            "Intercept",
            *MIXING_PREDICTORS,
            "1|window_idx_sigma",
            "1|clade_sigma",
        ],
        odds_ratio_vars=MIXING_PREDICTORS,
        categorical=MIXING_GROUP_VARS,
        **SAMPLING_KWARGS,
    )

    print("\nFitting joint expanded mixing model")
    joint_expanded_results["joint_expanded"] = fit_and_summarise_model(
        data=mixing_expanded_df,
        formula=joint_mixing_expanded_formula,
        var_names=[
            "Intercept",
            *MIXING_PREDICTORS,
            *EXPANDED_ADJUSTERS,
            "1|window_idx_sigma",
            "1|clade_sigma",
        ],
        odds_ratio_vars=MIXING_PREDICTORS,
        categorical=MIXING_GROUP_VARS,
        **SAMPLING_KWARGS,
    )
else:
    print("RUN_MIXING_GRID=False; skipping joint mixing-model sampling.")
